# 02. Ekstraksi Fitur NLP

Model tidak dapat membaca teks secara langsung, sehingga berita perlu diubah menjadi angka (fitur).
Digunakan dua pendekatan:
1. Leksikon sentimen VADER dan Loughran-McDonald: memberi skor sentimen berdasarkan daftar kata
   yang sudah baku.
2. TF-IDF: memberi bobot pada setiap kata berdasarkan tingkat kepentingannya pada hari tersebut,
   kemudian diringkas menjadi 20 komponen.

Setiap pendekatan diterapkan pada tiga bagian berita agar dapat dibandingkan:
judul (`title`), paragraf awal (`lead`), dan isi lengkap (`full`).

Hasil ekstraksi disimpan di `data/features.csv`.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

DATA = Path("../data") if Path("../data").exists() else Path("data")  # berjalan dari folder notebook atau folder utama repo
SOURCES = {"title": "title", "lead": "lead", "full": "content"}  # nama fitur : kolom teks
N_SVD = 20

art = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in sorted((DATA / "articles").glob("*.csv"))])
art = art.fillna("").sort_values("date", kind="stable").reset_index(drop=True)
days = pd.concat([pd.read_csv(DATA / f"{s}.csv", usecols=["date"], parse_dates=["date"])
                  for s in ["train", "val", "test"]])["date"].sort_values().reset_index(drop=True)
train_days = pd.read_csv(DATA / "train.csv", usecols=["date"], parse_dates=["date"])["date"]
print(f"{len(art)} artikel, {len(days)} hari trading")

14213 artikel, 1179 hari trading


## 1. Memuat Leksikon VADER dan Loughran-McDonald
Dua leksikon sentimen yang digunakan:
- VADER (Hutto & Gilbert, 2014): digunakan melalui paket resmi `vaderSentiment`. Setiap kata memiliki nilai
  sentimen antara -4 (sangat negatif) dan +4 (sangat positif), dengan aturan untuk negasi, kata penguat,
  kata "but", huruf kapital, dan tanda baca.
- Loughran-McDonald (Loughran & McDonald, 2011): kamus kata berdasarkan laporan keuangan, dengan kategori
  negative, positive, uncertainty, litigious, strong_modal, weak_modal, dan constraining.

File kamus Loughran-McDonald tersimpan di folder `data/lexicon/`.

In [2]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()
LEX_DIR = DATA / "lexicon"

# Loughran-McDonald: kata masuk kategori jika nilai kolom kategorinya > 0 (tahun kata ditambahkan)
lm_dict = pd.read_csv(LEX_DIR / "Loughran-McDonald_MasterDictionary_1993-2025.csv", keep_default_na=False)
LM_CATS = ["Negative", "Positive", "Uncertainty", "Litigious", "Strong_Modal", "Weak_Modal", "Constraining"]
lm_sets = {c.lower(): set(lm_dict.loc[lm_dict[c] > 0, "Word"].str.upper()) for c in LM_CATS}

print(f"VADER: {len(vader.lexicon):,} entri")
print("Loughran-McDonald:", {k: len(v) for k, v in lm_sets.items()})

VADER: 7,506 entri
Loughran-McDonald: {'negative': 2345, 'positive': 347, 'uncertainty': 297, 'litigious': 903, 'strong_modal': 19, 'weak_modal': 27, 'constraining': 184}


## 2. Cara Menghitung Skor
VADER dirancang untuk teks pendek, sehingga skor dihitung per kalimat lalu dirata-ratakan per artikel.
Hasilnya terdiri atas `compound` (rentang -1 sampai 1) serta proporsi `pos`, `neg`, dan `neu`.

Loughran-McDonald dihitung sebagai jumlah kata per kategori per 1.000 kata, sehingga berita yang panjang
tidak otomatis memperoleh nilai lebih besar. Kata positif yang didahului negasi dalam jarak tiga kata
dihitung sebagai negatif. Skor `lm_tone` = (positif - negatif) / (positif + negatif), dengan rentang -1 sampai 1.

In [3]:
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
LM_NEGATORS = {"NO", "NOT", "NONE", "NEITHER", "NEVER", "NOBODY"}


def score_text(text):
    """Skor VADER (rata-rata per kalimat) dan Loughran-McDonald untuk satu artikel."""
    sents = [s for s in SENT_SPLIT.split(text) if s.strip()]
    if sents:
        v = pd.DataFrame([vader.polarity_scores(s) for s in sents]).mean()
        out = {"vader_compound": v["compound"], "vader_pos": v["pos"], "vader_neg": v["neg"], "vader_neu": v["neu"]}
    else:
        out = {"vader_compound": 0.0, "vader_pos": 0.0, "vader_neg": 0.0, "vader_neu": 0.0}

    tokens = re.findall(r"[A-Z]+", text.upper().replace("’", "").replace("'", ""))
    count = dict.fromkeys(lm_sets, 0)
    for i, tok in enumerate(tokens):
        for cat, words in lm_sets.items():
            if tok in words:
                if cat == "positive" and LM_NEGATORS & set(tokens[max(0, i - 3): i]):
                    count["negative"] += 1                     # positif yang dinegasi dihitung negatif
                else:
                    count[cat] += 1
    n = max(len(tokens), 1)
    out.update({f"lm_{c}": 1000 * k / n for c, k in count.items()})
    pn = count["positive"] + count["negative"]
    out["lm_tone"] = (count["positive"] - count["negative"]) / pn if pn else 0.0
    return out


# Contoh penghitungan skor pada tiga kalimat
for s in ["Stocks rally as inflation cools and investors turn optimistic",
          "Oil prices surge after the missile attack",
          "Investors fear a deeper recession as the crisis spreads"]:
    sc = score_text(s)
    print(f"{s[:60]:60s} VADER={sc['vader_compound']:+.2f}  LM tone={sc['lm_tone']:+.2f}")

Stocks rally as inflation cools and investors turn optimisti VADER=+0.32  LM tone=+1.00
Oil prices surge after the missile attack                    VADER=-0.48  LM tone=+0.00
Investors fear a deeper recession as the crisis spreads      VADER=-0.88  LM tone=-1.00


## 3. Skor Leksikon Harian
Skor dihitung untuk setiap artikel, kemudian dirata-ratakan per hari. Selain itu ditambahkan fitur berikut:
- `log_n_news` dan `has_news`: jumlah berita pada hari tersebut dan penanda ada tidaknya berita.
- Fitur perubahan (akhiran `_dev`): nilai hari ini dikurangi rata-rata lima hari sebelumnya.
  Fitur ini digunakan karena pasar cenderung bereaksi terhadap perubahan pemberitaan yang mendadak.

In [4]:
LEX_COLS = ["vader_compound", "vader_pos", "vader_neg", "vader_neu"] + [f"lm_{c}" for c in lm_sets] + ["lm_tone"]


def daily_lexicon(src, col):
    scores = pd.DataFrame([score_text(t) for t in art[col]])
    scores["date"] = art["date"].values
    daily = scores.groupby("date")[LEX_COLS].mean().reindex(days).fillna(0.0)
    n_news = art.groupby("date").size().reindex(days).fillna(0)
    daily["log_n_news"] = np.log1p(n_news.values)
    daily["has_news"] = (n_news.values > 0).astype(int)
    for c in ["vader_compound", "lm_tone", "lm_negative", "lm_uncertainty"]:
        daily[f"{c}_dev"] = daily[c] - daily[c].shift(1).rolling(5, min_periods=1).mean().fillna(0)
    daily.columns = [f"lex_{src}_{c}" for c in daily.columns]
    return daily


lex = pd.concat([daily_lexicon(s, c) for s, c in SOURCES.items()], axis=1)
print("Jumlah fitur leksikon:", lex.shape[1], f"({lex.shape[1] // 3} per sumber teks)")
lex.filter(like="lex_lead_").describe().T[["mean", "std", "min", "max"]].round(3)

Jumlah fitur leksikon: 54 (18 per sumber teks)


,mean,std,min,max
lex_lead_vader_compound,0.031,0.101,-0.356,0.361
lex_lead_vader_pos,0.073,0.022,0.000,0.191
lex_lead_vader_neg,0.060,0.022,0.000,0.167
lex_lead_vader_neu,0.853,0.104,0.000,1.000
lex_lead_lm_negative,22.467,9.036,0.000,75.758
lex_lead_lm_positive,7.451,4.187,0.000,27.778
lex_lead_lm_uncertainty,8.428,4.452,0.000,30.303
lex_lead_lm_litigious,4.259,4.844,0.000,38.371
lex_lead_lm_strong_modal,4.467,3.295,0.000,24.525
lex_lead_lm_weak_modal,4.938,3.335,0.000,29.484


## 4. TF-IDF dan TruncatedSVD
Seluruh berita dalam satu hari digabungkan menjadi satu dokumen. TF-IDF kemudian memberi bobot tinggi pada kata
yang sering muncul pada hari tersebut tetapi jarang muncul pada hari lain. Stopword bahasa Inggris (misalnya
"the" dan "and") serta kata yang muncul kurang dari lima hari dihapus.

Hasil TF-IDF terdiri atas ribuan kolom, terlalu banyak untuk data train yang hanya 825 hari. Oleh karena itu,
hasilnya diringkas menjadi 20 komponen menggunakan TruncatedSVD (Latent Semantic Analysis), yang mengelompokkan
kata-kata yang sering muncul bersamaan.

TF-IDF dan SVD hanya dilatih menggunakan data train agar kosakata dari periode validation dan test tidak ikut digunakan.

In [5]:
tfidf_parts, info = [], {}
is_train = days.isin(train_days).values

for src, col in SOURCES.items():
    docs = art.groupby("date")[col].apply(" ".join).reindex(days).fillna("")
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.5, sublinear_tf=True, stop_words="english",
                          token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b", max_features=30000, dtype=np.float32)
    svd = TruncatedSVD(n_components=N_SVD, random_state=42)
    svd.fit(vec.fit_transform(docs[is_train]))           # dilatih di train saja
    z = svd.transform(vec.transform(docs))               # diterapkan ke semua hari
    tfidf_parts.append(pd.DataFrame(z, index=days, columns=[f"tfidf_{src}_svd{i:02d}" for i in range(N_SVD)]))

    words = np.array(vec.get_feature_names_out())
    info[src] = {"ukuran_kosakata": len(words),
                 "variansi_20_komponen": f"{svd.explained_variance_ratio_.sum():.1%}",
                 "kata_teratas_komponen_2": ", ".join(words[np.argsort(-svd.components_[1])[:6]])}

tfidf = pd.concat(tfidf_parts, axis=1)
pd.DataFrame(info).T

,ukuran_kosakata,variansi_20_komponen,kata_teratas_komponen_2
title,3804,7.9%,"stock price, real time, check, real, price, stock"
lead,12787,7.3%,"ukraine, russia, russian, invasion, putin, inv..."
full,30000,8.9%,"rewards, annual fee, fee, card, bonus, select"


Tabel di atas menampilkan kata-kata utama pada komponen kedua. Pada isi lengkap, komponen ini didominasi kata
terkait kartu kredit seperti "rewards", "annual fee", dan "card" (konten afiliasi CNBC Select). Pada judul,
komponen ini didominasi frasa "stock price" dan "real time". Hal ini menunjukkan bahwa data masih memuat
berita yang tidak berkaitan dengan geopolitik.

## 5. Menyimpan Fitur

In [6]:
features = pd.concat([lex, tfidf], axis=1)
features.index.name = "date"
features.reset_index().to_csv(DATA / "features.csv", index=False, date_format="%Y-%m-%d")
print("Tersimpan data/features.csv:", features.shape)

Tersimpan data/features.csv: (1179, 114)
